In [1]:
!pip install -q timm safetensors torch torchvision numpy requests matplotlib scipy tqdm pandas psutil


# Plain Federated Learning Baseline

The previous notebooks focused on what a curious server can infer from local or aggregated updates. We now remove the attacker entirely and study the normal federated learning process.

The goal of this notebook is to establish the **clean FL baseline** that later attack and defense experiments should be compared against.

We start from the same centrally fine-tuned **MobileNetV4 Conv Small 0.5** checkpoint used throughout the previous stages. Ten simulated clients hold different local QuickDraw datasets. Clients train locally, the server combines their classifier-head parameters with FedAvg, and the resulting global model is evaluated after every communication round.

There is no malicious client, no poisoning, no inversion procedure, and no privacy defense in this notebook.


## Experimental configuration

The clean baseline uses the following setup:

- **Global initialization:** the Stage 1 QuickDraw checkpoint based on `mobilenetv4_conv_small_050.e3000_r224_in1k`.
- **Clients:** 10 simulated users.
- **Rounds:** 50.
- **Participation:** all 10 clients participate in every round.
- **Local optimization:** one local epoch with Adam.
- **Trainable parameters:** classifier head only. The MobileNetV4 backbone stays frozen.
- **Aggregation:** sample-weighted FedAvg.
- **Client size:** 1,000 to 2,000 samples.
- **Client label coverage:** 5 to 10 of the 10 available classes.
- **Client split:** 90% local training and 10% local validation.
- **Non-IID label skew:** class proportions are sampled with a Dirichlet distribution using $\alpha=0.35$.
- **Evaluation:** weighted client train/validation metrics plus the centralized Stage 1 held-out test set.

The client data are drawn from QuickDraw indices starting at 10,000, away from the first 5,000 examples per class used to construct the original centralized train/validation/test split. This reduces accidental overlap with the Stage 1 test set.


## Imports, reproducibility, and experiment configuration


In [2]:
import copy
import csv
import json
import math
import os
import random
import urllib.request
import threading
import time

import psutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import timm
import timm.data as timm_data
from safetensors.torch import load_file as load_safetensors
from safetensors.torch import save_file as save_safetensors

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {DEVICE}")

# Stage 1 artifacts. These are produced by the centralized fine-tuning notebook.
STAGE1_OUTPUT_DIR = Path("./outputs")
STAGE1_CONFIG_PATH = STAGE1_OUTPUT_DIR / "config.json"

# Raw QuickDraw data location.
DATA_DIR = Path("./data")

# Output directory for this clean FL run.
OUTPUT_DIR = Path("./outputs_federated_no_attack_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Human-readable label -> QuickDraw dataset class name, as used in .npy filenames.
SELECTED_CLASSES = {
    "line": "line",
    "circle": "circle",
    "square": "square",
    "triangle": "triangle",
    "star": "star",
    "tshirt": "t-shirt",
    "smiley_face": "smiley_face",
    "sun": "sun",
    "cloud": "cloud",
    "donut": "donut",
}
CLASS_NAMES = list(SELECTED_CLASSES.keys())
QUICKDRAW_FILE_NAMES = list(SELECTED_CLASSES.values())
NUM_CLASSES = len(CLASS_NAMES)

# Original Stage 1 split configuration. The global test set comes from this split.
MAX_ITEMS_PER_CLASS = 5_000
TRAIN_PCT = 0.8
VAL_PCT = 0.1
TEST_PCT = 0.1
assert abs(TRAIN_PCT + VAL_PCT + TEST_PCT - 1.0) < 1e-9

# Federated data construction.
NUM_CLIENTS = 10
CLIENT_MIN_SAMPLES = 1_000
CLIENT_MAX_SAMPLES = 2_000
CLIENT_MIN_CLASSES = 5
CLIENT_MAX_CLASSES = NUM_CLASSES
CLIENT_TRAIN_PCT = 0.90
CLIENT_VAL_PCT = 0.10
DIRICHLET_ALPHA = 0.35
MIN_SAMPLES_PER_SELECTED_CLASS = 20

# Client data are deliberately drawn from later QuickDraw indices, away from the Stage 1
# first-5,000 split. This avoids accidental overlap with the centralized test set.
CLIENT_RAW_START_IDX = 10_000
HARD_CANDIDATES_PER_CLASS = 5_000
HARD_POOL_MULTIPLIER = 2.0

# Federated learning configuration.
NUM_ROUNDS = 50
CLIENT_FRACTION = 1.0          # 1.0 means all clients participate in every round.
LOCAL_EPOCHS = 1
LOCAL_BATCH_SIZE = 128
LOCAL_LR = 1e-3
LOCAL_WEIGHT_DECAY = 0.0

# Batch sizes for feature extraction / evaluation.
FORWARD_BATCH_SIZE = 256
# Keep DataLoader multiprocessing disabled by default.
# This script is often executed from Colab/Jupyter. Repeatedly creating DataLoaders with
# num_workers > 0 in that environment can emit noisy worker-shutdown assertions such as
# "can only test a child process". Using 0 workers is slightly less parallel but much more
# robust and does not change the experiment.
NUM_WORKERS = 0
PIN_MEMORY = DEVICE.type == "cuda"

def start_resource_monitor(device: torch.device, sample_interval_s: float = 0.05) -> Dict:
    """Start scoped wall-time, process-RAM, and CUDA-VRAM monitoring."""
    process = psutil.Process(os.getpid())
    baseline_rss = process.memory_info().rss
    rss_samples = [baseline_rss]
    stop_event = threading.Event()

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        baseline_cuda_allocated = torch.cuda.memory_allocated(device)
        baseline_cuda_reserved = torch.cuda.memory_reserved(device)
    else:
        baseline_cuda_allocated = None
        baseline_cuda_reserved = None

    def sample_ram():
        while not stop_event.is_set():
            rss_samples.append(process.memory_info().rss)
            stop_event.wait(sample_interval_s)

    thread = threading.Thread(target=sample_ram, daemon=True)
    thread.start()

    return {
        "process": process,
        "baseline_rss": baseline_rss,
        "rss_samples": rss_samples,
        "stop_event": stop_event,
        "thread": thread,
        "start_time": time.perf_counter(),
        "baseline_cuda_allocated": baseline_cuda_allocated,
        "baseline_cuda_reserved": baseline_cuda_reserved,
    }


def stop_resource_monitor(monitor: Dict, device: torch.device) -> Dict:
    """Stop monitoring and return JSON-serializable runtime and memory statistics."""
    if device.type == "cuda":
        torch.cuda.synchronize(device)

    elapsed_s = time.perf_counter() - monitor["start_time"]
    monitor["stop_event"].set()
    monitor["thread"].join(timeout=1.0)
    monitor["rss_samples"].append(monitor["process"].memory_info().rss)

    peak_rss = max(monitor["rss_samples"])
    baseline_rss = monitor["baseline_rss"]

    stats = {
        "fl_loop_elapsed_s": float(elapsed_s),
        "baseline_process_ram_mib": float(baseline_rss / (1024 ** 2)),
        "peak_process_ram_mib": float(peak_rss / (1024 ** 2)),
        "peak_process_ram_increase_mib": float((peak_rss - baseline_rss) / (1024 ** 2)),
        "ram_sampling_interval_s": 0.05,
        "peak_cuda_vram_allocated_mib": None,
        "peak_cuda_vram_reserved_mib": None,
        "peak_cuda_vram_allocated_increase_mib": None,
        "peak_cuda_vram_reserved_increase_mib": None,
    }

    if device.type == "cuda":
        peak_allocated = torch.cuda.max_memory_allocated(device)
        peak_reserved = torch.cuda.max_memory_reserved(device)
        stats.update({
            "peak_cuda_vram_allocated_mib": float(peak_allocated / (1024 ** 2)),
            "peak_cuda_vram_reserved_mib": float(peak_reserved / (1024 ** 2)),
            "peak_cuda_vram_allocated_increase_mib": float(
                (peak_allocated - monitor["baseline_cuda_allocated"]) / (1024 ** 2)
            ),
            "peak_cuda_vram_reserved_increase_mib": float(
                (peak_reserved - monitor["baseline_cuda_reserved"]) / (1024 ** 2)
            ),
        })

    return stats


def print_resource_stats(stats: Dict) -> None:
    print(f"Full FL loop time including round evaluation: {stats['fl_loop_elapsed_s']:.2f} s")
    print(
        f"Peak process RAM (RSS): {stats['peak_process_ram_mib']:.1f} MiB "
        f"(+{stats['peak_process_ram_increase_mib']:.1f} MiB over FL-loop start)"
    )
    if stats["peak_cuda_vram_allocated_mib"] is not None:
        print(
            f"Peak CUDA VRAM allocated: {stats['peak_cuda_vram_allocated_mib']:.1f} MiB "
            f"(+{stats['peak_cuda_vram_allocated_increase_mib']:.1f} MiB)"
        )
        print(
            f"Peak CUDA VRAM reserved: {stats['peak_cuda_vram_reserved_mib']:.1f} MiB "
            f"(+{stats['peak_cuda_vram_reserved_increase_mib']:.1f} MiB)"
        )
    else:
        print(f"CUDA VRAM: not applicable on device={DEVICE}")


Using device: cuda


## QuickDraw data and the original global test set

The notebook downloads the same 10 QuickDraw bitmap classes used in the centralized stage.

The original Stage 1 split is reconstructed deterministically from the first 5,000 samples per class. Only its **test partition** is used as the centralized reference test set during federated training.

This gives us two complementary views of model quality:

1. performance on the heterogeneous client data participating in FL,
2. performance on the fixed centralized test set used before FL started.


In [3]:
def get_quickdraw_class_names() -> List[str]:
    """Full official QuickDraw category list, used only for optional validation."""
    url = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return [x.strip() for x in r.text.splitlines() if x.strip()]



def download_quickdraw_dataset(root: Path, class_names: Optional[List[str]] = None):
    """Download the numpy bitmap (.npy) files for the requested QuickDraw class names."""
    root.mkdir(parents=True, exist_ok=True)
    base_url = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

    print("Downloading QuickDraw dataset files used by the Stage 1 label space...")
    for class_name in tqdm(class_names):
        fpath = root / f"{class_name}.npy"
        if fpath.exists():
            continue
        url = f"{base_url}{class_name.replace('_', '%20').replace('-', '%2D')}.npy"
        try:
            urllib.request.urlretrieve(url, fpath)
        except Exception as e:
            raise RuntimeError(f"Failed to download class '{class_name}' from {url}: {e}")
    print("Download complete.")



def load_quickdraw_arrays(root: Path, file_names: List[str], max_items_per_class: int):
    """Load raw (N, 784) uint8 bitmaps and integer labels for the given classes.

    Labels are assigned by the index in `file_names`, i.e. label order equals CLASS_NAMES.
    This reproduces the original Stage 1 train/validation/test split construction.
    """
    X_list, y_list = [], []
    print(f"Loading up to {max_items_per_class} examples per class for the Stage 1 split...")
    for label_idx, fname in enumerate(tqdm(file_names)):
        fpath = root / f"{fname}.npy"
        data = np.load(fpath, mmap_mode="r")
        n = min(max_items_per_class, data.shape[0])
        print(f"Label id: {label_idx} - Label name: {CLASS_NAMES[label_idx]} ({fname}).")
        print(f"\tNumber of samples in class: {data.shape[0]} - Loaded {n} samples.")
        X_list.append(np.array(data[:n]))
        y_list.append(np.full(n, label_idx, dtype=np.int64))

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X, y



def stratified_split_indices(labels: np.ndarray, train_pct: float, val_pct: float, test_pct: float, seed: int = SEED):
    """Per-class stratified split. Returns index arrays into `labels`."""
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []

    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)
        n = len(cls_idx)
        n_train = int(round(n * train_pct))
        n_val = int(round(n * val_pct))
        train_idx.append(cls_idx[:n_train])
        val_idx.append(cls_idx[n_train:n_train + n_val])
        test_idx.append(cls_idx[n_train + n_val:])

    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    test_idx = np.concatenate(test_idx)

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


## Frozen-backbone feature extraction

The FL experiment updates only the classifier head. The MobileNetV4 feature extractor never changes during the 50 communication rounds.

For this reason, running every 224x224 image through the frozen backbone again in every local epoch would perform the same expensive computation repeatedly. We instead compute the backbone embeddings once and store them as tensors.

Local client training then operates directly on these embeddings.

For a completely frozen deterministic backbone, optimizing the same classifier head from the same embeddings produces the same head-level optimization problem as repeatedly forwarding the images through that frozen backbone.


In [4]:
class QuickDrawRawDataset(Dataset):
    """Dataset wrapper for raw QuickDraw numpy bitmap rows.

    Each item is converted from a flattened 28x28 uint8 bitmap to a PIL image, replicated to
    3 channels, and transformed with the same timm preprocessing used by the Stage 1 model.
    """

    def __init__(self, X_flat: np.ndarray, y: np.ndarray, transform):
        self.X_flat = X_flat
        self.y = y.astype(np.int64)
        self.transform = transform

    def __len__(self):
        return int(self.X_flat.shape[0])

    def __getitem__(self, idx):
        img_28 = self.X_flat[idx].reshape(28, 28).astype(np.uint8)
        img_rgb = np.stack([img_28, img_28, img_28], axis=-1)
        pil_img = Image.fromarray(img_rgb)
        x = self.transform(pil_img)
        y = int(self.y[idx])
        return x, y


@torch.no_grad()
def embedding_of(model: nn.Module, images_3ch: torch.Tensor) -> torch.Tensor:
    """Return pooled frozen-backbone embeddings for normalized model inputs."""
    feats = model.forward_features(images_3ch)
    pooled = model.forward_head(feats, pre_logits=True)
    return pooled


@torch.no_grad()
def compute_embeddings(
    model: nn.Module,
    X_flat: np.ndarray,
    y: np.ndarray,
    transform,
    batch_size: int = FORWARD_BATCH_SIZE,
    desc: str = "Computing embeddings",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Precompute frozen-backbone embeddings and labels as CPU tensors."""
    ds = QuickDrawRawDataset(X_flat, y, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model.eval()
    all_emb, all_y = [], []
    for xb, yb in tqdm(loader, desc=desc):
        xb = xb.to(DEVICE, non_blocking=True)
        emb = embedding_of(model, xb).detach().cpu()
        all_emb.append(emb)
        all_y.append(yb.cpu())

    return torch.cat(all_emb, dim=0), torch.cat(all_y, dim=0)


@torch.no_grad()
def compute_losses_for_raw_images(
    model: nn.Module,
    X_flat: np.ndarray,
    y: np.ndarray,
    transform,
    batch_size: int = FORWARD_BATCH_SIZE,
    desc: str = "Scoring hard examples",
) -> np.ndarray:
    """Compute per-sample cross-entropy loss of the Stage 1 model on raw bitmaps."""
    ds = QuickDrawRawDataset(X_flat, y, transform)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model.eval()
    losses = []
    for xb, yb in tqdm(loader, desc=desc, leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        logits = model(xb)
        batch_losses = F.cross_entropy(logits, yb, reduction="none")
        losses.append(batch_losses.detach().cpu().numpy())
    return np.concatenate(losses, axis=0)


## Construct heterogeneous client datasets

The client population is non-IID.

Each client receives a randomly chosen dataset size between 1,000 and 2,000 examples and observes between 5 and 10 classes. A minimum number of examples is first assigned to every selected class. The remaining examples follow a low-concentration Dirichlet draw with $\alpha=0.35$, producing substantial differences in class proportions across clients.

The examples are also selected from a candidate pool using their cross-entropy loss under the Stage 1 model. Higher loss examples are retained first. This gives the FL stage a more demanding local distribution than simply sampling easy or arbitrary QuickDraw examples.

Each client's resulting dataset is split into 90% local training and 10% local validation data using a stratified split.


In [5]:
@dataclass
class ClientRawData:
    client_id: int
    class_counts: Dict[int, int]
    train_X: np.ndarray
    train_y: np.ndarray
    val_X: np.ndarray
    val_y: np.ndarray
    train_embeddings: Optional[torch.Tensor] = None
    train_labels: Optional[torch.Tensor] = None
    val_embeddings: Optional[torch.Tensor] = None
    val_labels: Optional[torch.Tensor] = None

    @property
    def n_train(self) -> int:
        return int(self.train_y.shape[0])

    @property
    def n_val(self) -> int:
        return int(self.val_y.shape[0])

    @property
    def n_total(self) -> int:
        return self.n_train + self.n_val



def allocate_counts_for_client(rng: np.random.Generator, n_samples: int, selected_classes: np.ndarray) -> Dict[int, int]:
    """Allocate a non-IID class-count vector for one client.

    A small floor is assigned to each selected class so the client truly contains all selected
    labels. The remaining samples are distributed using a low-alpha Dirichlet draw to create
    label skew.
    """
    k = len(selected_classes)
    min_floor = min(MIN_SAMPLES_PER_SELECTED_CLASS, max(1, n_samples // (5 * k)))
    base = np.full(k, min_floor, dtype=np.int64)
    remaining = int(n_samples - base.sum())
    if remaining < 0:
        raise ValueError("Client sample count is too small for the selected class floor.")

    if remaining > 0:
        proportions = rng.dirichlet(np.full(k, DIRICHLET_ALPHA))
        extra = rng.multinomial(remaining, proportions)
    else:
        extra = np.zeros(k, dtype=np.int64)

    counts = base + extra
    return {int(cls): int(count) for cls, count in zip(selected_classes, counts) if count > 0}



def build_client_plans(seed: int = SEED) -> List[Dict[int, int]]:
    """Create non-IID client class-count plans before sampling actual images."""
    rng = np.random.default_rng(seed)
    plans = []

    for client_id in range(NUM_CLIENTS):
        n_samples = int(rng.integers(CLIENT_MIN_SAMPLES, CLIENT_MAX_SAMPLES + 1))
        n_classes = int(rng.integers(CLIENT_MIN_CLASSES, CLIENT_MAX_CLASSES + 1))
        selected_classes = rng.choice(NUM_CLASSES, size=n_classes, replace=False)
        counts = allocate_counts_for_client(rng, n_samples, selected_classes)
        plans.append(counts)

    return plans



def class_demands_from_plans(plans: List[Dict[int, int]]) -> np.ndarray:
    """Total number of raw images required for each class across all clients."""
    demands = np.zeros(NUM_CLASSES, dtype=np.int64)
    for counts in plans:
        for cls, n in counts.items():
            demands[cls] += n
    return demands



def select_hard_examples_for_class(
    class_idx: int,
    fname: str,
    demand: int,
    model: nn.Module,
    transform,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Select `demand` hard raw QuickDraw examples for one class.

    Candidate images are drawn from raw indices >= CLIENT_RAW_START_IDX, scored by the Stage 1
    model's cross-entropy loss, sorted by loss, and the hardest examples are retained. The
    retained set is shuffled before client assignment so that one client does not receive all
    of the single hardest images of a class.
    """
    fpath = DATA_DIR / f"{fname}.npy"
    data = np.load(fpath, mmap_mode="r")

    available = data.shape[0] - CLIENT_RAW_START_IDX
    if available <= 0:
        raise RuntimeError(f"Class {fname} has no data after index {CLIENT_RAW_START_IDX}.")

    candidate_size = int(max(HARD_CANDIDATES_PER_CLASS, math.ceil(HARD_POOL_MULTIPLIER * demand)))
    candidate_size = min(candidate_size, available)
    if demand > candidate_size:
        raise RuntimeError(
            f"Need {demand} samples for class {CLASS_NAMES[class_idx]}, but candidate pool has only "
            f"{candidate_size}. Increase HARD_CANDIDATES_PER_CLASS or reduce client sizes."
        )

    raw_indices = rng.choice(np.arange(CLIENT_RAW_START_IDX, data.shape[0]), size=candidate_size, replace=False)
    X_candidates = np.array(data[raw_indices])
    y_candidates = np.full(candidate_size, class_idx, dtype=np.int64)

    losses = compute_losses_for_raw_images(
        model,
        X_candidates,
        y_candidates,
        transform,
        desc=f"Scoring hard examples for {CLASS_NAMES[class_idx]}",
    )

    order = np.argsort(-losses)
    selected_order = order[:demand]

    selected_X = X_candidates[selected_order]
    selected_raw_indices = raw_indices[selected_order]
    selected_losses = losses[selected_order]

    # Shuffle the selected hard pool before distributing it to clients.
    shuffle_order = rng.permutation(demand)
    return selected_X[shuffle_order], selected_raw_indices[shuffle_order], selected_losses[shuffle_order]



def stratified_client_train_val_split(X: np.ndarray, y: np.ndarray, seed: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Create a per-client stratified 90/10 train/validation split."""
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []

    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        n_cls = len(cls_idx)
        n_val = max(1, int(round(n_cls * CLIENT_VAL_PCT))) if n_cls >= 2 else 0
        val_idx.append(cls_idx[:n_val])
        train_idx.append(cls_idx[n_val:])

    train_idx = np.concatenate(train_idx) if len(train_idx) > 0 else np.array([], dtype=np.int64)
    val_idx = np.concatenate(val_idx) if len(val_idx) > 0 else np.array([], dtype=np.int64)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]



def build_clients_from_hard_examples(
    plans: List[Dict[int, int]],
    model: nn.Module,
    transform,
    seed: int = SEED,
) -> List[ClientRawData]:
    """Materialize client datasets according to non-IID plans and hard-example selection."""
    rng = np.random.default_rng(seed + 1000)
    demands = class_demands_from_plans(plans)

    print("\nClient class-count plans:")
    for client_id, counts in enumerate(plans):
        readable = ", ".join([f"{CLASS_NAMES[c]}:{n}" for c, n in sorted(counts.items())])
        print(f"  Client {client_id:02d}: n={sum(counts.values())}, classes={len(counts)} | {readable}")

    print("\nTotal class demands across clients:")
    for c, demand in enumerate(demands):
        print(f"  {CLASS_NAMES[c]:12s}: {int(demand)}")

    hard_pools: Dict[int, Dict[str, np.ndarray]] = {}
    for class_idx, demand in enumerate(demands):
        if demand == 0:
            continue
        selected_X, selected_raw_indices, selected_losses = select_hard_examples_for_class(
            class_idx=class_idx,
            fname=QUICKDRAW_FILE_NAMES[class_idx],
            demand=int(demand),
            model=model,
            transform=transform,
            rng=rng,
        )
        hard_pools[class_idx] = {
            "X": selected_X,
            "raw_indices": selected_raw_indices,
            "losses": selected_losses,
            "cursor": np.array([0], dtype=np.int64),
        }
        print(
            f"Selected {demand} hard examples for {CLASS_NAMES[class_idx]:12s} | "
            f"loss range: {selected_losses.min():.4f}--{selected_losses.max():.4f}, "
            f"mean={selected_losses.mean():.4f}"
        )

    clients = []
    for client_id, counts in enumerate(plans):
        X_parts, y_parts = [], []
        for class_idx, n in sorted(counts.items()):
            pool = hard_pools[class_idx]
            start = int(pool["cursor"][0])
            end = start + int(n)
            pool["cursor"][0] = end

            X_parts.append(pool["X"][start:end])
            y_parts.append(np.full(int(n), class_idx, dtype=np.int64))

        X_client = np.concatenate(X_parts, axis=0)
        y_client = np.concatenate(y_parts, axis=0)

        shuffle_order = rng.permutation(len(y_client))
        X_client = X_client[shuffle_order]
        y_client = y_client[shuffle_order]

        train_X, train_y, val_X, val_y = stratified_client_train_val_split(
            X_client,
            y_client,
            seed=seed + 2000 + client_id,
        )

        clients.append(
            ClientRawData(
                client_id=client_id,
                class_counts=counts,
                train_X=train_X,
                train_y=train_y,
                val_X=val_X,
                val_y=val_y,
            )
        )
        print(
            f"Client {client_id:02d}: train={len(train_y)}, val={len(val_y)}, "
            f"total={len(y_client)}"
        )

    return clients


## Local training and FedAvg

At the start of every communication round, each selected client receives the current global classifier head.

The client trains its local copy for one epoch with Adam. After local training, the server computes sample-weighted FedAvg:

$$\theta_{t+1}=\sum_{k \in S_t}\frac{n_k}{\sum_{j \in S_t} n_j}\theta_{t+1}^{(k)}.$$

Here, $n_k$ is the number of local training samples for client $k$.

All clients participate in every round in this baseline, so the experiment does not include partial participation or availability effects. Those can be introduced later as separate FL-system variables.

Because only the head is trainable, FedAvg is applied only to the classifier-head parameters.


In [6]:
class LinearHead(nn.Module):
    """Small wrapper around the classifier head so it can be trained independently."""

    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        return self.classifier(embeddings)



def make_head_from_model(model: nn.Module) -> LinearHead:
    """Copy the current timm classifier head into a standalone LinearHead module."""
    classifier = model.get_classifier()
    if not isinstance(classifier, nn.Linear):
        raise TypeError(f"Expected a linear classifier head, got: {type(classifier)}")

    head = LinearHead(classifier.in_features, classifier.out_features)
    head.classifier.load_state_dict(copy.deepcopy(classifier.state_dict()))
    return head



def load_head_into_model(model: nn.Module, head: LinearHead):
    """Copy standalone head weights back into the timm model's classifier."""
    classifier = model.get_classifier()
    classifier.load_state_dict(copy.deepcopy(head.classifier.state_dict()))



def train_local_head(global_head: LinearHead, embeddings: torch.Tensor, labels: torch.Tensor) -> LinearHead:
    """Train a local copy of the classifier head on one client's train embeddings."""
    local_head = copy.deepcopy(global_head).to(DEVICE)
    local_head.train()

    ds = TensorDataset(embeddings, labels)
    loader = DataLoader(ds, batch_size=LOCAL_BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.Adam(local_head.parameters(), lr=LOCAL_LR, weight_decay=LOCAL_WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()

    for _epoch in range(LOCAL_EPOCHS):
        for emb, yb in loader:
            emb = emb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = local_head(emb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    return local_head.cpu()



def fedavg_heads(local_heads: List[LinearHead], weights: List[int]) -> LinearHead:
    """Weighted FedAvg over standalone classifier heads."""
    assert len(local_heads) == len(weights) and len(local_heads) > 0
    total_weight = float(sum(weights))

    avg_head = copy.deepcopy(local_heads[0])
    avg_state = avg_head.state_dict()

    for key in avg_state:
        avg_state[key] = torch.zeros_like(avg_state[key])

    for head, weight in zip(local_heads, weights):
        state = head.state_dict()
        coeff = float(weight) / total_weight
        for key in avg_state:
            avg_state[key] += coeff * state[key]

    avg_head.load_state_dict(avg_state)
    return avg_head


@torch.no_grad()
def evaluate_head_on_embeddings(head: LinearHead, embeddings: torch.Tensor, labels: torch.Tensor, batch_size: int = 2048) -> Tuple[float, float]:
    """Evaluate a classifier head on precomputed embeddings. Returns loss and accuracy."""
    head = copy.deepcopy(head).to(DEVICE)
    head.eval()

    ds = TensorDataset(embeddings, labels)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total_correct = 0
    total = 0
    for emb, yb in loader:
        emb = emb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        logits = head(emb)
        total_loss += float(criterion(logits, yb).item())
        total_correct += int((logits.argmax(dim=1) == yb).sum().item())
        total += int(yb.numel())

    return total_loss / max(total, 1), 100.0 * total_correct / max(total, 1)


@torch.no_grad()
def evaluate_weighted_client_metrics(head: LinearHead, clients: List[ClientRawData], split: str) -> Tuple[float, float]:
    """Weighted average client metrics for train or validation split.

    This mirrors FedAvg weighting: losses are averaged by sample count, and accuracy is the
    sample-weighted aggregate accuracy across clients.
    """
    total_loss_sum = 0.0
    total_correct = 0.0
    total_count = 0

    head_eval = copy.deepcopy(head).to(DEVICE)
    head_eval.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")

    for client in clients:
        if split == "train":
            embeddings, labels = client.train_embeddings, client.train_labels
        elif split == "val":
            embeddings, labels = client.val_embeddings, client.val_labels
        else:
            raise ValueError("split must be 'train' or 'val'.")

        ds = TensorDataset(embeddings, labels)
        loader = DataLoader(ds, batch_size=2048, shuffle=False)
        for emb, yb in loader:
            emb = emb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            logits = head_eval(emb)
            total_loss_sum += float(criterion(logits, yb).item())
            total_correct += int((logits.argmax(dim=1) == yb).sum().item())
            total_count += int(yb.numel())

    return total_loss_sum / max(total_count, 1), 100.0 * total_correct / max(total_count, 1)


## Metrics and reporting

After each round, the notebook evaluates:

- sample-weighted client training loss and accuracy,
- sample-weighted client validation loss and accuracy,
- loss and accuracy on the fixed centralized Stage 1 test set.

The notebook also stores the complete per-round history and produces:

- client label-distribution figures,
- FL convergence figures,
- CSV history,
- JSON configuration and summary files,
- the final federated model in `.safetensors` format.

For engineering comparison with later notebooks, the FL loop additionally reports:

- cumulative time spent in local fine-tuning and aggregation across all rounds,
- full FL-loop time including round-level evaluation,
- peak process RAM,
- peak CUDA allocated and reserved VRAM when CUDA is available.

Dataset download, hard-example construction, and one-time embedding precomputation are intentionally excluded from the fine-tuning-time figure.


In [7]:
def save_client_distribution_figure(clients: List[ClientRawData], output_dir: Path):
    """Save client x class count matrix as CSV and heatmap figure."""
    matrix = np.zeros((len(clients), NUM_CLASSES), dtype=np.int64)
    for client in clients:
        for cls, count in client.class_counts.items():
            matrix[client.client_id, cls] = count

    df = pd.DataFrame(matrix, index=[f"client_{c.client_id:02d}" for c in clients], columns=CLASS_NAMES)
    df["total"] = df.sum(axis=1)
    df.to_csv(output_dir / "client_label_distribution.csv")

    fig, ax = plt.subplots(figsize=(12, 5.5))
    im = ax.imshow(matrix, aspect="auto")
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(clients)))
    ax.set_yticklabels([f"Client {c.client_id:02d}\n(n={c.n_total})" for c in clients])
    ax.set_title("Federated client-label distribution (non-IID hard examples)")
    ax.set_xlabel("Class")
    ax.set_ylabel("Client")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j] > 0:
                ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=7)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Number of samples")
    fig.tight_layout()
    fig.savefig(output_dir / "client_label_distribution.png", dpi=200)
    fig.savefig(output_dir / "client_label_distribution.pdf")
    plt.close(fig)



def save_convergence_figure(history: List[Dict], output_dir: Path):
    """Save convergence curves for train/val/test loss and accuracy."""
    rounds = [row["round"] for row in history]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(rounds, [row["train_loss"] for row in history], marker="o", label="Client train")
    axes[0].plot(rounds, [row["val_loss"] for row in history], marker="o", label="Client val")
    axes[0].plot(rounds, [row["test_loss"] for row in history], marker="o", label="Global test")
    axes[0].set_title("Federated fine-tuning loss")
    axes[0].set_xlabel("Round")
    axes[0].set_ylabel("Cross-entropy loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(rounds, [row["train_acc"] for row in history], marker="o", label="Client train")
    axes[1].plot(rounds, [row["val_acc"] for row in history], marker="o", label="Client val")
    axes[1].plot(rounds, [row["test_acc"] for row in history], marker="o", label="Global test")
    axes[1].set_title("Federated fine-tuning accuracy")
    axes[1].set_xlabel("Round")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_dir / "fl_convergence.png", dpi=200)
    fig.savefig(output_dir / "fl_convergence.pdf")
    plt.close(fig)



def save_history_csv(history: List[Dict], output_dir: Path):
    """Save per-round metrics to CSV."""
    path = output_dir / "fl_history.csv"
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)
    print(f"Saved FL history to: {path}")

def save_client_datasets(clients: List[ClientRawData], output_dir: Path) -> Path:
    """Persist the exact client datasets so later robustness notebooks reuse them unchanged."""
    dataset_dir = output_dir / "client_datasets"
    dataset_dir.mkdir(parents=True, exist_ok=True)

    manifest = {
        "num_clients": len(clients),
        "class_names": CLASS_NAMES,
        "quickdraw_file_names": QUICKDRAW_FILE_NAMES,
        "client_train_pct": CLIENT_TRAIN_PCT,
        "client_val_pct": CLIENT_VAL_PCT,
        "files": [],
    }

    for client in clients:
        class_count_vec = np.zeros(NUM_CLASSES, dtype=np.int64)
        for cls_idx, count in client.class_counts.items():
            class_count_vec[int(cls_idx)] = int(count)

        client_path = dataset_dir / f"client_{client.client_id:02d}.npz"
        np.savez_compressed(
            client_path,
            train_X=client.train_X.astype(np.uint8),
            train_y=client.train_y.astype(np.int64),
            val_X=client.val_X.astype(np.uint8),
            val_y=client.val_y.astype(np.int64),
            class_counts=class_count_vec,
        )

        manifest["files"].append({
            "client_id": int(client.client_id),
            "path": str(client_path),
            "n_train": int(client.n_train),
            "n_val": int(client.n_val),
            "n_total": int(client.n_total),
            "class_counts": {
                CLASS_NAMES[i]: int(class_count_vec[i])
                for i in range(NUM_CLASSES)
                if int(class_count_vec[i]) > 0
            },
        })

    manifest_path = dataset_dir / "manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Saved reusable client datasets to: {dataset_dir}")
    print(f"Saved client manifest to: {manifest_path}")
    return dataset_dir


## Run the clean federated learning experiment

The experiment first loads and verifies the Stage 1 checkpoint. It explicitly checks that the configured architecture is the same **MobileNetV4 Conv Small 0.5** model used in the previous notebooks and prints the instantiated parameter count.

The client datasets and frozen-backbone embeddings are then prepared once. Round 0 records model quality before any federated update.

For rounds 1 through 50:

1. every client receives the current global head,
2. every client trains locally for one epoch,
3. the server aggregates all local heads using sample-weighted FedAvg,
4. the new global head is evaluated on client and centralized test data.

No attack or defense mechanism is active.


In [8]:
def main():
    print(f"Number of classes: {NUM_CLASSES}")
    print(CLASS_NAMES)
    print(f"DataLoader config: num_workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}")

    # -------------------------------------------------------------------------
    # Load Stage 1 config and checkpoint.
    # -------------------------------------------------------------------------
    if not STAGE1_CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {STAGE1_CONFIG_PATH}. Run the Stage 1 centralized fine-tuning notebook first."
        )

    with open(STAGE1_CONFIG_PATH, "r") as f:
        stage1_config = json.load(f)

    MODEL_NAME = stage1_config["model_name"]
    EXPECTED_MODEL_NAME = "mobilenetv4_conv_small_050.e3000_r224_in1k"
    assert MODEL_NAME == EXPECTED_MODEL_NAME, (
        f"This notebook expects the FLTA Small MobileNetV4 checkpoint ({EXPECTED_MODEL_NAME}), "
        f"but config.json specifies {MODEL_NAME}."
    )
    STAGE1_CLASS_NAMES = stage1_config["class_names"]
    STAGE1_QUICKDRAW_FILE_NAMES = stage1_config["quickdraw_file_names"]
    IMG_SIZE = stage1_config["input_size"][-1]
    MEAN = stage1_config["mean"]
    STD = stage1_config["std"]
    STAGE1_NUM_CLASSES = stage1_config["num_classes"]

    assert STAGE1_CLASS_NAMES == CLASS_NAMES, (
        "Stage 1 class order does not match this script. Keep the label order identical."
    )
    assert STAGE1_QUICKDRAW_FILE_NAMES == QUICKDRAW_FILE_NAMES, (
        "Stage 1 QuickDraw filename order does not match this script."
    )
    assert STAGE1_NUM_CLASSES == NUM_CLASSES

    candidate_ckpts = sorted(STAGE1_OUTPUT_DIR.glob("*.safetensors"))
    if len(candidate_ckpts) == 0:
        raise FileNotFoundError(f"No .safetensors checkpoint found in {STAGE1_OUTPUT_DIR}.")
    STAGE1_CKPT_PATH = candidate_ckpts[0]

    print(f"Stage 1 model: {MODEL_NAME}")
    print(f"Stage 1 classes ({NUM_CLASSES}): {CLASS_NAMES}")
    print(f"Stage 1 checkpoint: {STAGE1_CKPT_PATH}")

    global_model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
    num_model_params = sum(p.numel() for p in global_model.parameters())
    print(f"Model parameters: {num_model_params:,}")
    stage1_state_dict = load_safetensors(str(STAGE1_CKPT_PATH))
    missing, unexpected = global_model.load_state_dict(stage1_state_dict, strict=True)
    print(f"Loaded Stage 1 weights. Missing: {missing} | Unexpected: {unexpected}")

    global_model.to(DEVICE)
    global_model.eval()
    for p in global_model.parameters():
        p.requires_grad = False

    classifier = global_model.get_classifier()
    print(f"Classifier head: {classifier}")

    # -------------------------------------------------------------------------
    # Data download and transforms.
    # -------------------------------------------------------------------------
    download_quickdraw_dataset(DATA_DIR, QUICKDRAW_FILE_NAMES)
    print("Available data files:", sorted(os.listdir(DATA_DIR)))

    data_config = {
        "input_size": (3, IMG_SIZE, IMG_SIZE),
        "mean": MEAN,
        "std": STD,
    }
    eval_transform = timm_data.create_transform(**data_config, is_training=False)
    print(eval_transform)

    # -------------------------------------------------------------------------
    # Original Stage 1 test set.
    # -------------------------------------------------------------------------
    X_all, y_all = load_quickdraw_arrays(DATA_DIR, QUICKDRAW_FILE_NAMES, MAX_ITEMS_PER_CLASS)
    train_idx, val_idx, test_idx = stratified_split_indices(y_all, TRAIN_PCT, VAL_PCT, TEST_PCT, seed=SEED)
    print(f"Original split sizes | Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

    X_test = X_all[test_idx]
    y_test = y_all[test_idx]

    test_embeddings, test_labels = compute_embeddings(
        global_model,
        X_test,
        y_test,
        eval_transform,
        desc="Precomputing global test embeddings",
    )

    # -------------------------------------------------------------------------
    # Build non-IID hard-example clients.
    # -------------------------------------------------------------------------
    client_plans = build_client_plans(seed=SEED)
    clients = build_clients_from_hard_examples(client_plans, global_model, eval_transform, seed=SEED)
    save_client_distribution_figure(clients, OUTPUT_DIR)
    print(f"Saved client distribution figure to: {OUTPUT_DIR / 'client_label_distribution.png'}")
    if len(clients) != NUM_CLIENTS:
      raise RuntimeError(f"Expected {NUM_CLIENTS} clients, built {len(clients)}.")

    save_client_datasets(clients, OUTPUT_DIR)

    # Precompute embeddings for all client data.
    for client in clients:
        client.train_embeddings, client.train_labels = compute_embeddings(
            global_model,
            client.train_X,
            client.train_y,
            eval_transform,
            desc=f"Client {client.client_id:02d} train embeddings",
        )
        client.val_embeddings, client.val_labels = compute_embeddings(
            global_model,
            client.val_X,
            client.val_y,
            eval_transform,
            desc=f"Client {client.client_id:02d} val embeddings",
        )

    # -------------------------------------------------------------------------
    # Initial global metrics before any federated fine-tuning.
    # -------------------------------------------------------------------------
    global_head = make_head_from_model(global_model)

    history: List[Dict] = []

    def record_metrics(
        round_idx: int,
        head: LinearHead,
        selected_client_ids: Optional[List[int]] = None,
        round_finetune_time_s: Optional[float] = None,
    ):
        train_loss, train_acc = evaluate_weighted_client_metrics(head, clients, split="train")
        val_loss, val_acc = evaluate_weighted_client_metrics(head, clients, split="val")
        test_loss, test_acc = evaluate_head_on_embeddings(head, test_embeddings, test_labels)

        row = {
            "round": int(round_idx),
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "test_loss": float(test_loss),
            "test_acc": float(test_acc),
            "selected_clients": "all" if selected_client_ids is None else ",".join(map(str, selected_client_ids)),
            "round_finetune_time_s": None if round_finetune_time_s is None else float(round_finetune_time_s),
        }
        history.append(row)
        print(
            f"[Round {round_idx:02d}] "
            f"train loss={train_loss:.4f}, acc={train_acc:.2f}% | "
            f"val loss={val_loss:.4f}, acc={val_acc:.2f}% | "
            f"test loss={test_loss:.4f}, acc={test_acc:.2f}%"
        )

    print("\nInitial metrics before federated fine-tuning:")
    record_metrics(round_idx=0, head=global_head, selected_client_ids=None)

    # -------------------------------------------------------------------------
    # FedAvg rounds.
    # -------------------------------------------------------------------------
    rng_round = np.random.default_rng(SEED + 5000)
    num_selected = max(1, int(round(CLIENT_FRACTION * NUM_CLIENTS)))

    print(f"\nStarting clean FedAvg for {NUM_ROUNDS} rounds...")
    print(
        f"Local training: head-only Adam | epochs={LOCAL_EPOCHS}, batch={LOCAL_BATCH_SIZE}, "
        f"lr={LOCAL_LR}, weight_decay={LOCAL_WEIGHT_DECAY}"
    )

    # Reset the training RNG here so the clean baseline can be reproduced
    # by later controlled attack notebooks that reuse these exact clients.
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    fl_monitor = start_resource_monitor(DEVICE)
    round_finetune_times = []

    for round_idx in range(1, NUM_ROUNDS + 1):
        round_finetune_start = time.perf_counter()

        if num_selected == NUM_CLIENTS:
            selected_clients = clients
        else:
            selected_ids = rng_round.choice(NUM_CLIENTS, size=num_selected, replace=False)
            selected_clients = [clients[int(i)] for i in selected_ids]

        local_heads = []
        local_weights = []
        for client in selected_clients:
            local_head = train_local_head(global_head, client.train_embeddings, client.train_labels)
            local_heads.append(local_head)
            local_weights.append(client.n_train)

        global_head = fedavg_heads(local_heads, local_weights)
        round_finetune_time_s = time.perf_counter() - round_finetune_start
        round_finetune_times.append(round_finetune_time_s)

        selected_ids_print = [c.client_id for c in selected_clients]
        record_metrics(
            round_idx=round_idx,
            head=global_head,
            selected_client_ids=selected_ids_print,
            round_finetune_time_s=round_finetune_time_s,
        )

    fl_resource_stats = stop_resource_monitor(fl_monitor, DEVICE)
    federated_finetuning_time_s = float(sum(round_finetune_times))

    print("\nFederated fine-tuning resource summary:")
    print(f"Fine-tuning + aggregation time across {NUM_ROUNDS} rounds: {federated_finetuning_time_s:.2f} s")
    print_resource_stats(fl_resource_stats)

    # -------------------------------------------------------------------------
    # Save final full model and config.
    # -------------------------------------------------------------------------
    load_head_into_model(global_model, global_head)
    final_state = {k: v.detach().cpu().contiguous() for k, v in global_model.state_dict().items()}
    model_path = OUTPUT_DIR / "federated_head_only_model.safetensors"
    save_safetensors(final_state, str(model_path))
    print(f"\nSaved federated fine-tuned model to: {model_path}")

    config = {
        "model_name": MODEL_NAME,
        "class_names": CLASS_NAMES,
        "quickdraw_file_names": QUICKDRAW_FILE_NAMES,
        "num_classes": NUM_CLASSES,
        "input_size": [3, IMG_SIZE, IMG_SIZE],
        "mean": MEAN,
        "std": STD,
        "stage1_checkpoint": str(STAGE1_CKPT_PATH),
        "federated_training": {
            "num_clients": NUM_CLIENTS,
            "num_rounds": NUM_ROUNDS,
            "client_fraction": CLIENT_FRACTION,
            "local_optimizer": "Adam",
            "local_epochs": LOCAL_EPOCHS,
            "local_batch_size": LOCAL_BATCH_SIZE,
            "local_lr": LOCAL_LR,
            "local_weight_decay": LOCAL_WEIGHT_DECAY,
            "finetune_strategy": "head_only_frozen_backbone",
            "client_min_samples": CLIENT_MIN_SAMPLES,
            "client_max_samples": CLIENT_MAX_SAMPLES,
            "client_min_classes": CLIENT_MIN_CLASSES,
            "client_max_classes": CLIENT_MAX_CLASSES,
            "client_train_pct": CLIENT_TRAIN_PCT,
            "client_val_pct": CLIENT_VAL_PCT,
            "dirichlet_alpha": DIRICHLET_ALPHA,
            "client_raw_start_idx": CLIENT_RAW_START_IDX,
            "hard_candidates_per_class": HARD_CANDIDATES_PER_CLASS,
        },
        "initial_metrics": history[0],
        "final_metrics": history[-1],
        "resource_metrics": {
            "federated_finetuning_time_s": federated_finetuning_time_s,
            **fl_resource_stats,
        },
    }
    config_path = OUTPUT_DIR / "config.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Saved FL config to: {config_path}")

    save_history_csv(history, OUTPUT_DIR)
    save_convergence_figure(history, OUTPUT_DIR)
    print(f"Saved convergence figure to: {OUTPUT_DIR / 'fl_convergence.png'}")

    summary = {
        "initial": history[0],
        "final": history[-1],
        "resource_metrics": {
            "federated_finetuning_time_s": federated_finetuning_time_s,
            **fl_resource_stats,
        },
        "delta": {
            "train_loss": history[-1]["train_loss"] - history[0]["train_loss"],
            "train_acc": history[-1]["train_acc"] - history[0]["train_acc"],
            "val_loss": history[-1]["val_loss"] - history[0]["val_loss"],
            "val_acc": history[-1]["val_acc"] - history[0]["val_acc"],
            "test_loss": history[-1]["test_loss"] - history[0]["test_loss"],
            "test_acc": history[-1]["test_acc"] - history[0]["test_acc"],
        },
    }
    with open(OUTPUT_DIR / "fl_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    latex_summary = f"""% Clean federated fine-tuning summary generated by flta_federated_no_attack_head_only.py
\\begin{{table}}[h]
\\centering
\\caption{{Clean head-only federated fine-tuning before and after FedAvg.}}
\\label{{tab:clean-fl-summary}}
\\begin{{tabular}}{{lcccc}}
\\toprule
Metric split & Initial loss & Initial acc. (\\%) & Final loss & Final acc. (\\%) \\\\
\\midrule
Client train & {history[0]['train_loss']:.4f} & {history[0]['train_acc']:.2f} & {history[-1]['train_loss']:.4f} & {history[-1]['train_acc']:.2f} \\\\
Client validation & {history[0]['val_loss']:.4f} & {history[0]['val_acc']:.2f} & {history[-1]['val_loss']:.4f} & {history[-1]['val_acc']:.2f} \\\\
Global test & {history[0]['test_loss']:.4f} & {history[0]['test_acc']:.2f} & {history[-1]['test_loss']:.4f} & {history[-1]['test_acc']:.2f} \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}
"""
    with open(OUTPUT_DIR / "fl_latex_summary.txt", "w") as f:
        f.write(latex_summary)

    print("\n================================================================================")
    print("Clean FL summary")
    print("================================================================================")
    print(latex_summary)


if __name__ == "__main__":
    main()


Number of classes: 10
['line', 'circle', 'square', 'triangle', 'star', 'tshirt', 'smiley_face', 'sun', 'cloud', 'donut']
DataLoader config: num_workers=0, pin_memory=True
Stage 1 model: mobilenetv4_conv_small_050.e3000_r224_in1k
Stage 1 classes (10): ['line', 'circle', 'square', 'triangle', 'star', 'tshirt', 'smiley_face', 'sun', 'cloud', 'donut']
Stage 1 checkpoint: outputs/mobilenetv4_small_quickdraw10.safetensors
Model parameters: 970,762
Loaded Stage 1 weights. Missing: [] | Unexpected: []
Classifier head: Linear(in_features=1280, out_features=10, bias=True)


  0%|          | 0/10 [00:00<?, ?it/s]

Download complete.
Available data files: ['circle.npy', 'cloud.npy', 'donut.npy', 'line.npy', 'smiley_face.npy', 'square.npy', 'star.npy', 'sun.npy', 't-shirt.npy', 'triangle.npy']
Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)
Loading up to 5000 examples per class for the Stage 1 split...


  0%|          | 0/10 [00:00<?, ?it/s]

Label id: 0 - Label name: line (line).
	Number of samples in class: 143549 - Loaded 5000 samples.
Label id: 1 - Label name: circle (circle).
	Number of samples in class: 122876 - Loaded 5000 samples.
Label id: 2 - Label name: square (square).
	Number of samples in class: 125145 - Loaded 5000 samples.
Label id: 3 - Label name: triangle (triangle).
	Number of samples in class: 123170 - Loaded 5000 samples.
Label id: 4 - Label name: star (star).
	Number of samples in class: 137619 - Loaded 5000 samples.
Label id: 5 - Label name: tshirt (t-shirt).
	Number of samples in class: 125233 - Loaded 5000 samples.
Label id: 6 - Label name: smiley_face (smiley_face).
	Number of samples in class: 124386 - Loaded 5000 samples.
Label id: 7 - Label name: sun (sun).
	Number of samples in class: 133781 - Loaded 5000 samples.
Label id: 8 - Label name: cloud (cloud).
	Number of samples in class: 120265 - Loaded 5000 samples.
Label id: 9 - Label name: donut (donut).
	Number of samples in class: 140751 - Load

...

Client 08 train embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

Client 08 val embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Client 09 train embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

Client 09 val embeddings:   0%|          | 0/1 [00:00<?, ?it/s]


Initial metrics before federated fine-tuning:
[Round 00] train loss=0.3876, acc=90.34% | val loss=0.4305, acc=89.67% | test loss=0.0925, acc=97.66%

Starting clean FedAvg for 50 rounds...
Local training: head-only Adam | epochs=1, batch=128, lr=0.001, weight_decay=0.0
[Round 01] train loss=0.3750, acc=90.74% | val loss=0.4163, acc=89.74% | test loss=0.0888, acc=97.74%
[Round 02] train loss=0.3710, acc=90.93% | val loss=0.4139, acc=89.67% | test loss=0.0882, acc=97.74%
[Round 03] train loss=0.3692, acc=90.94% | val loss=0.4142, acc=89.74% | test loss=0.0884, acc=97.68%
[Round 04] train loss=0.3670, acc=90.98% | val loss=0.4123, acc=89.67% | test loss=0.0879, acc=97.66%
[Round 05] train loss=0.3653, acc=90.98% | val loss=0.4106, acc=89.74% | test loss=0.0875, acc=97.64%
[Round 06] train loss=0.3663, acc=91.10% | val loss=0.4140, acc=89.60% | test loss=0.0884, acc=97.62%
[Round 07] train loss=0.3648, acc=91.08% | val loss=0.4122, acc=89.45% | test loss=0.0880, acc=97.64%
[Round 08] train